<a href="https://colab.research.google.com/github/Levan-Danelia/FRTB/blob/main/FRTB_CRVG_Securitizations_Non_ACTP.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# Cell 1: Setup and Initial Data
# -----------------------------------------------------------------------------
# This cell loads the initial portfolio data and all necessary regulatory
# parameters for the calculation. The output displays the starting positions,
# which corresponds to Step 1 in the HTML report.

import pandas as pd
import numpy as np

# --- Initial Portfolio Data ---
# Represents the starting positions and their gross vega sensitivities.
portfolio_data = [
    {'position_id': 1, 'bucket': 1, 'credit_quality': 'IG', 'sector': 'RMBS-Prime', 'tranche': 'Tranche A', 'tenor_str': '1Y', 'gross_sensitivity': 80},
    {'position_id': 2, 'bucket': 1, 'credit_quality': 'IG', 'sector': 'RMBS-Prime', 'tranche': 'Tranche B', 'tenor_str': '3Y', 'gross_sensitivity': -60},
    {'position_id': 3, 'bucket': 12, 'credit_quality': 'NIG', 'sector': 'CMBS', 'tranche': 'Tranche C', 'tenor_str': '1Y', 'gross_sensitivity': 90},
    {'position_id': 4, 'bucket': 12, 'credit_quality': 'NIG', 'sector': 'CMBS', 'tranche': 'Tranche C', 'tenor_str': '5Y', 'gross_sensitivity': -70}
]

# --- Regulatory Parameters (Articles 325an, 325ao, 325ax, 325ay) ---
VEGA_RISK_WEIGHT = 1.00
RHO_DELTA_TRANCHE_DIFF = 0.40
RHO_DELTA_TRANCHE_SAME = 1.00
RHO_DELTA_TENOR_DIFF = 0.80
RHO_DELTA_TENOR_SAME = 1.00
ALPHA_OPTION_MATURITY = 0.01
GAMMA_CROSS_BUCKET = 0.0

# Create the initial DataFrame
df = pd.DataFrame(portfolio_data)
df['tenor_years'] = df['tenor_str'].str.replace('Y', '').astype(int)

print("--- Step 1: Identify Risk Factors ---")
print("Each unique combination of tranche, sector, quality, and tenor is a distinct risk factor.")
print("\n" + df[['position_id', 'bucket', 'credit_quality', 'sector', 'tranche', 'tenor_str']].to_string(index=False))

--- Step 1: Identify Risk Factors ---
Each unique combination of tranche, sector, quality, and tenor is a distinct risk factor.

 position_id  bucket credit_quality     sector   tranche tenor_str
           1       1             IG RMBS-Prime Tranche A        1Y
           2       1             IG RMBS-Prime Tranche B        3Y
           3      12            NIG       CMBS Tranche C        1Y
           4      12            NIG       CMBS Tranche C        5Y


In [3]:
# -----------------------------------------------------------------------------
# Cell 2: Net and Weighted Sensitivities
# -----------------------------------------------------------------------------
# This cell corresponds to Steps 3 and 4 in the HTML report.
# Step 3: Since all risk factors are unique, no netting is applied.
# Step 4: Net sensitivities are multiplied by the regulatory risk weight.

# Step 3: Net Sensitivities
# No netting is applicable as all risk factors are unique. Net = Gross.
df['net_sensitivity'] = df['gross_sensitivity']

# Step 4: Weighted Sensitivities
df['weighted_sensitivity'] = df['net_sensitivity'] * VEGA_RISK_WEIGHT

# Calculate Sb (sum of weighted sensitivities per bucket) for later steps
s_b = df.groupby('bucket')['weighted_sensitivity'].sum().to_dict()

print("\n--- Steps 3 & 4: Net and Weighted Sensitivities ---")
print("No netting was applied as risk factors are unique. Sensitivities are weighted.")
print("\n" + df[['bucket', 'tranche', 'tenor_str', 'net_sensitivity', 'weighted_sensitivity']].to_string(index=False))
print(f"\nSum of Weighted Sensitivities (S_b):")
print(f"  - S_b for Bucket 1: {s_b.get(1, 0):.2f}")
print(f"  - S_b for Bucket 12: {s_b.get(12, 0):.2f}")


--- Steps 3 & 4: Net and Weighted Sensitivities ---
No netting was applied as risk factors are unique. Sensitivities are weighted.

 bucket   tranche tenor_str  net_sensitivity  weighted_sensitivity
      1 Tranche A        1Y               80                  80.0
      1 Tranche B        3Y              -60                 -60.0
     12 Tranche C        1Y               90                  90.0
     12 Tranche C        5Y              -70                 -70.0

Sum of Weighted Sensitivities (S_b):
  - S_b for Bucket 1: 20.00
  - S_b for Bucket 12: 20.00


In [4]:
# -----------------------------------------------------------------------------
# Cell 3: Intra-Bucket Correlation Coefficients (Medium Scenario)
# -----------------------------------------------------------------------------
# This cell implements Step 5 from the HTML report. It calculates the
# correlation (rho_kl) for each pair of risk factors within the same bucket.

correlation_data = []
intra_bucket_correlations = {}
positions = df.to_dict('records')

# Group positions by bucket
positions_by_bucket = df.groupby('bucket')

for bucket_id, group in positions_by_bucket:
    group_positions = group.to_dict('records')
    # Iterate over all unique pairs within the bucket
    for i in range(len(group_positions)):
        for j in range(i + 1, len(group_positions)):
            pos1, pos2 = group_positions[i], group_positions[j]

            # 1. Calculate Delta Correlation component
            rho_tranche = RHO_DELTA_TRANCHE_SAME if pos1['tranche'] == pos2['tranche'] else RHO_DELTA_TRANCHE_DIFF
            rho_tenor = RHO_DELTA_TENOR_SAME if pos1['tenor_years'] == pos2['tenor_years'] else RHO_DELTA_TENOR_DIFF
            rho_delta = rho_tranche * rho_tenor

            # 2. Calculate Option Maturity Correlation component
            t1, t2 = pos1['tenor_years'], pos2['tenor_years']
            rho_maturity = np.exp(-ALPHA_OPTION_MATURITY * abs(t1 - t2) / min(t1, t2))

            # 3. Final Vega Correlation
            rho_kl = min(rho_delta * rho_maturity, 1.0)

            # Store for display and later calculations
            pair_key = (pos1['position_id'], pos2['position_id'])
            intra_bucket_correlations[pair_key] = rho_kl
            correlation_data.append([
                f"Bucket {bucket_id}",
                f"{pos1['tranche']} {pos1['tenor_str']}",
                f"{pos2['tranche']} {pos2['tenor_str']}",
                f"{rho_delta:.2%}",
                f"{rho_maturity:.2%}",
                f"{rho_kl:.2%}"
            ])

df_correlations = pd.DataFrame(correlation_data, columns=["Bucket", "Risk Factor k", "Risk Factor l", "Delta Corr", "Maturity Corr", "Final Corr (rho_kl)"])

print("\n--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---")
print("Correlations are based on delta parameters and option maturity differences.")
print("\n" + df_correlations.to_string(index=False))


--- Step 5: Intra-Bucket Correlation Coefficients (Medium Scenario) ---
Correlations are based on delta parameters and option maturity differences.

   Bucket Risk Factor k Risk Factor l Delta Corr Maturity Corr Final Corr (rho_kl)
 Bucket 1  Tranche A 1Y  Tranche B 3Y     32.00%        98.02%              31.37%
Bucket 12  Tranche C 1Y  Tranche C 5Y     80.00%        96.08%              76.86%


In [5]:
# -----------------------------------------------------------------------------
# Cell 4: Intra- and Cross-Bucket Aggregation (Medium Scenario)
# -----------------------------------------------------------------------------
# This cell implements Steps 7 and 8 from the HTML report for the Medium Scenario.
# Step 7: Calculate bucket-specific capital (K_b).
# Step 8: Aggregate K_b values to get the final capital charge.

k_b_values = {}
print("\n--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---")

# Group positions by bucket to perform aggregation
positions_by_bucket = df.groupby('bucket')

for bucket_id, group in positions_by_bucket:
    group_positions = group.to_dict('records')

    # Sum of squares of weighted sensitivities
    sum_ws_sq = np.sum([p['weighted_sensitivity']**2 for p in group_positions])

    # Sum of cross-products
    cross_term = 0
    for i in range(len(group_positions)):
        for j in range(i + 1, len(group_positions)):
            pos1, pos2 = group_positions[i], group_positions[j]
            pair_key = (pos1['position_id'], pos2['position_id'])
            rho_kl = intra_bucket_correlations[pair_key]
            cross_term += 2 * rho_kl * pos1['weighted_sensitivity'] * pos2['weighted_sensitivity']

    # Calculate K_b
    k_b = np.sqrt(max(0, sum_ws_sq + cross_term))
    k_b_values[bucket_id] = k_b

    print(f"\nCalculation for Bucket {bucket_id}:")
    print(f"  - Sum of Squares (Σ WS_k²): {sum_ws_sq:,.2f}")
    print(f"  - Sum of Cross-Products (Σ ρ_kl*WS_k*WS_l): {cross_term:,.2f}")
    print(f"  - Bucket Capital (K_{bucket_id}): {k_b:,.2f}")

# Step 8: Cross-Bucket Aggregation
print("\n--- Step 8: Cross-Bucket Aggregation (Medium Scenario) ---")
sum_k_b_sq = sum(k**2 for k in k_b_values.values())

# Cross-bucket term (is zero in this case as gamma = 0)
cross_bucket_term = 2 * GAMMA_CROSS_BUCKET * s_b.get(1, 0) * s_b.get(12, 0)

capital_medium = np.sqrt(sum_k_b_sq + cross_bucket_term)

print(f"\nFinal Calculation:")
print(f"  - Sum of K_b²: {sum_k_b_sq:,.2f}")
print(f"  - Cross-Bucket Term (γ*S_b*S_c): {cross_bucket_term:,.2f}")
print("----------------------------------------------------------")
print(f"  Medium Scenario Capital Requirement: {capital_medium:,.2f}")


--- Step 7: Intra-Bucket Aggregation (Medium Scenario) ---

Calculation for Bucket 1:
  - Sum of Squares (Σ WS_k²): 10,000.00
  - Sum of Cross-Products (Σ ρ_kl*WS_k*WS_l): -3,011.17
  - Bucket Capital (K_1): 83.60

Calculation for Bucket 12:
  - Sum of Squares (Σ WS_k²): 13,000.00
  - Sum of Cross-Products (Σ ρ_kl*WS_k*WS_l): -9,684.76
  - Bucket Capital (K_12): 57.58

--- Step 8: Cross-Bucket Aggregation (Medium Scenario) ---

Final Calculation:
  - Sum of K_b²: 10,304.07
  - Cross-Bucket Term (γ*S_b*S_c): 0.00
----------------------------------------------------------
  Medium Scenario Capital Requirement: 101.51


In [6]:
# -----------------------------------------------------------------------------
# Cell 5: High and Low Correlation Scenarios
# -----------------------------------------------------------------------------
# This cell implements Step 9 from the HTML report. It recalculates the
# total capital for High and Low correlation scenarios by adjusting the
# baseline correlation parameters.

def calculate_capital_for_scenario(correlations, s_b_dict):
    """Helper function to recalculate capital for a given set of correlations."""
    k_b_scenario = {}
    positions_by_bucket = df.groupby('bucket')

    # Recalculate K_b for each bucket with new correlations
    for bucket_id, group in positions_by_bucket:
        group_positions = group.to_dict('records')
        sum_ws_sq = np.sum([p['weighted_sensitivity']**2 for p in group_positions])
        cross_term = 0
        for i in range(len(group_positions)):
            for j in range(i + 1, len(group_positions)):
                pos1, pos2 = group_positions[i], group_positions[j]
                pair_key = (pos1['position_id'], pos2['position_id'])
                rho_kl = correlations.get(pair_key, 0)
                cross_term += 2 * rho_kl * pos1['weighted_sensitivity'] * pos2['weighted_sensitivity']
        k_b_scenario[bucket_id] = np.sqrt(max(0, sum_ws_sq + cross_term))

    # Recalculate final capital
    sum_k_b_sq = sum(k**2 for k in k_b_scenario.values())
    # Cross bucket gamma is 0, so stressed gamma is also 0
    total_capital = np.sqrt(sum_k_b_sq)
    return total_capital

# High Correlation Scenario
corrs_high = {k: min(v * 1.25, 1.0) for k, v in intra_bucket_correlations.items()}
capital_high = calculate_capital_for_scenario(corrs_high, s_b)

# Low Correlation Scenario
corrs_low = {k: max(2 * v - 1.0, 0.75 * v) for k, v in intra_bucket_correlations.items()}
capital_low = calculate_capital_for_scenario(corrs_low, s_b)

# --- Display Results ---
scenario_data = {
    'Scenario': ['Medium Correlation', 'High Correlation', 'Low Correlation'],
    'Vega Capital Requirement': [capital_medium, capital_high, capital_low]
}
df_scenarios = pd.DataFrame(scenario_data)

print("\n--- Step 9: Correlation Scenarios ---")
print("Capital is recalculated under stressed correlation assumptions.")
print("\nResulting Capital per Scenario:")
print(df_scenarios.round(2).to_string(index=False))


--- Step 9: Correlation Scenarios ---
Capital is recalculated under stressed correlation assumptions.

Resulting Capital per Scenario:
          Scenario  Vega Capital Requirement
Medium Correlation                    101.51
  High Correlation                     84.44
   Low Correlation                    116.10


In [7]:
# -----------------------------------------------------------------------------
# Cell 6: Final Charge Calculation
# -----------------------------------------------------------------------------
# This cell implements Step 10 from the HTML report. The final capital
# requirement is the maximum of the three calculated scenarios.

final_charge = max(capital_medium, capital_high, capital_low)
winning_scenario = df_scenarios.loc[df_scenarios['Vega Capital Requirement'].idxmax()]['Scenario']

print("\n--- Step 10: Final Charge Calculation ---")
print("The final requirement is the maximum capital charge from the three scenarios.")
print("\n-------------------------------------------------")
print(f" Final Vega Capital Requirement: {final_charge:,.2f}")
print(f" (Driven by the {winning_scenario})")
print("-------------------------------------------------")


--- Step 10: Final Charge Calculation ---
The final requirement is the maximum capital charge from the three scenarios.

-------------------------------------------------
 Final Vega Capital Requirement: 116.10
 (Driven by the Low Correlation)
-------------------------------------------------
